# 📊 MojoPQC-SCA: Results Visualization & Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hardikxk/pqc-sca/blob/main/notebooks/results_visualization.ipynb)

---

This notebook provides **publication-quality figures** and detailed analysis of the MojoPQC-SCA experimental results.

Run this **after** the training notebook (`colab_train.ipynb`) or upload pre-computed results.

> **No GPU required** — this notebook only loads and visualizes saved results.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
!pip install -q numpy pandas matplotlib seaborn h5py

import os
REPO_DIR = '/content/MojoPQC-SCA'
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
else:
    !git clone https://github.com/hardikxk/pqc-sca.git {REPO_DIR}
    os.chdir(REPO_DIR)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

# Publication-quality matplotlib settings
matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'font.family': 'serif',
    'axes.grid': True,
    'grid.alpha': 0.2,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = {
    'cnn': '#FF5722',
    'mps': '#2196F3',
    'accent': '#E91E63',
    'success': '#4CAF50',
    'dark': '#263238',
}

print(f'Working directory: {os.getcwd()}')
print(f'Results available: {os.path.isdir("results")}')

---

## 📈 Training Dynamics

In [ ]:
# ── Training curve comparison ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

logs = {}
for name, path, color in [
    ('CNN (baseline)', 'results/logs/cnn_training.csv', COLORS['cnn']),
    ('CNN+MPS (ours)', 'results/logs/cnn_mps_training.csv', COLORS['mps']),
]:
    if Path(path).exists():
        df = pd.read_csv(path)
        logs[name] = (df, color)

if logs:
    for name, (df, color) in logs.items():
        # Loss
        axes[0, 0].plot(df['epoch'], df['train_loss'], label=name,
                        color=color, linewidth=2)
        # Train accuracy
        axes[0, 1].plot(df['epoch'], df['train_accuracy'] * 100,
                        label=f'{name} (train)', color=color,
                        linewidth=2, linestyle='--', alpha=0.6)
        axes[0, 1].plot(df['epoch'], df['validation_accuracy'] * 100,
                        label=f'{name} (val)', color=color, linewidth=2)
        # Epoch time
        axes[1, 0].bar(df['epoch'] + (0.3 if 'MPS' in name else -0.3),
                       df['epoch_time_seconds'], width=0.5, label=name,
                       color=color, alpha=0.7)
        # Overfitting gap
        gap = (df['train_accuracy'] - df['validation_accuracy']) * 100
        axes[1, 1].plot(df['epoch'], gap, label=name, color=color, linewidth=2)

    axes[0, 0].set_title('Training Loss', fontweight='bold')
    axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Cross-Entropy Loss')
    axes[0, 0].legend()

    axes[0, 1].set_title('Accuracy', fontweight='bold')
    axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend(fontsize=8)

    axes[1, 0].set_title('Epoch Training Time', fontweight='bold')
    axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Time (seconds)')
    axes[1, 0].legend()

    axes[1, 1].set_title('Generalization Gap (train − val accuracy)', fontweight='bold')
    axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Gap (%)')
    axes[1, 1].axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    axes[1, 1].legend()

    plt.suptitle('Training Dynamics Comparison', fontweight='bold', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.savefig('results/figures/training_dynamics_full.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  No training logs found. Run colab_train.ipynb first.')

---

## 🔑 Guessing Entropy Analysis

The guessing entropy (GE) curve shows how the correct key's rank decreases as more attack traces are processed.

In [ ]:
# ── Guessing Entropy curve ────────────────────────────────────────
ge_path = Path('results/benchmarks/ge_results.json')

if ge_path.exists():
    ge_data = json.loads(ge_path.read_text())
    traces_n = np.array([int(k) for k in ge_data.keys()])
    ge_vals  = np.array(list(ge_data.values()))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Linear scale
    ax1.plot(traces_n, ge_vals, 'o-', color=COLORS['accent'], linewidth=2,
             markersize=5, markerfacecolor='white', markeredgewidth=1.5)
    ax1.axhline(y=1, color=COLORS['success'], linestyle='--', linewidth=1.5,
                label='Full key recovery (GE=1)')
    ax1.fill_between(traces_n, ge_vals, alpha=0.1, color=COLORS['accent'])
    ax1.set_xlabel('Number of Attack Traces')
    ax1.set_ylabel('Guessing Entropy')
    ax1.set_title('GE Curve (Linear)', fontweight='bold')
    ax1.legend()

    # Log scale
    ax2.semilogy(traces_n, ge_vals, 's-', color=COLORS['mps'], linewidth=2,
                 markersize=5, markerfacecolor='white', markeredgewidth=1.5)
    ax2.axhline(y=1, color=COLORS['success'], linestyle='--', linewidth=1.5,
                label='Full key recovery (GE=1)')
    ax2.set_xlabel('Number of Attack Traces')
    ax2.set_ylabel('Guessing Entropy (log scale)')
    ax2.set_title('GE Curve (Logarithmic)', fontweight='bold')
    ax2.legend()

    plt.suptitle('CNN+MPS Key Recovery Analysis', fontweight='bold', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('results/figures/ge_analysis_full.png', dpi=200, bbox_inches='tight')
    plt.show()

    # Statistics
    print('\n📊 GE Statistics:')
    print(f'   Initial GE  : {ge_vals[0]:.1f} (at {traces_n[0]} traces)')
    print(f'   Final GE    : {ge_vals[-1]:.1f} (at {traces_n[-1]} traces)')
    print(f'   Min GE      : {ge_vals.min():.1f}')
    recovery = traces_n[ge_vals <= 1]
    if len(recovery) > 0:
        print(f'   Key recovery: {recovery[0]} traces (GE ≤ 1)')
    else:
        print(f'   Key recovery: Not yet achieved (need more traces)')
else:
    print('⚠️  GE results not found.')

---

## 🏆 Model Comparison Dashboard

In [ ]:
# ── Comprehensive model comparison ────────────────────────────────
comp_path = Path('results/benchmarks/model_comparison.json')

if comp_path.exists():
    data = json.loads(comp_path.read_text())
    cnn_d, mps_d = data[0], data[1]

    fig = plt.figure(figsize=(16, 12))

    # ── 1. Parameter comparison ──
    ax1 = fig.add_subplot(2, 3, 1)
    bars = ax1.bar(['CNN', 'CNN+MPS'], [cnn_d['parameters'], mps_d['parameters']],
                   color=[COLORS['cnn'], COLORS['mps']], edgecolor='white', linewidth=2)
    ax1.set_title('Parameters', fontweight='bold')
    ax1.set_ylabel('Count')
    for bar, val in zip(bars, [cnn_d['parameters'], mps_d['parameters']]):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')

    # ── 2. Validation accuracy ──
    ax2 = fig.add_subplot(2, 3, 2)
    accs = [(cnn_d.get('validation_accuracy') or 0)*100,
            (mps_d.get('validation_accuracy') or 0)*100]
    bars = ax2.bar(['CNN', 'CNN+MPS'], accs,
                   color=[COLORS['cnn'], COLORS['mps']], edgecolor='white', linewidth=2)
    ax2.set_title('Validation Accuracy', fontweight='bold')
    ax2.set_ylabel('Accuracy (%)')
    for bar, val in zip(bars, accs):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

    # ── 3. GE comparison ──
    ax3 = fig.add_subplot(2, 3, 3)
    ge_labels = ['GE@100', 'GE@500', 'GE@all']
    cnn_ge = [cnn_d.get('ge_100', 0) or 0, cnn_d.get('ge_500', 0) or 0, cnn_d.get('ge_all', 0)]
    mps_ge = [mps_d.get('ge_100', 0) or 0, mps_d.get('ge_500', 0) or 0, mps_d.get('ge_all', 0)]
    x = np.arange(len(ge_labels))
    ax3.bar(x - 0.2, cnn_ge, 0.35, label='CNN', color=COLORS['cnn'], edgecolor='white')
    ax3.bar(x + 0.2, mps_ge, 0.35, label='CNN+MPS', color=COLORS['mps'], edgecolor='white')
    ax3.set_xticks(x); ax3.set_xticklabels(ge_labels)
    ax3.set_title('Guessing Entropy (lower = better)', fontweight='bold')
    ax3.set_ylabel('GE (key rank)')
    ax3.legend()

    # ── 4. Inference time ──
    ax4 = fig.add_subplot(2, 3, 4)
    times = [cnn_d['inference_seconds'], mps_d['inference_seconds']]
    bars = ax4.bar(['CNN', 'CNN+MPS'], times,
                   color=[COLORS['cnn'], COLORS['mps']], edgecolor='white', linewidth=2)
    ax4.set_title('Inference Time (attack set)', fontweight='bold')
    ax4.set_ylabel('Seconds')
    for bar, val in zip(bars, times):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}s', ha='center', fontsize=10, fontweight='bold')

    # ── 5. Memory usage ──
    ax5 = fig.add_subplot(2, 3, 5)
    mems = [cnn_d['peak_memory_bytes']/1e6, mps_d['peak_memory_bytes']/1e6]
    bars = ax5.bar(['CNN', 'CNN+MPS'], mems,
                   color=[COLORS['cnn'], COLORS['mps']], edgecolor='white', linewidth=2)
    ax5.set_title('Peak Memory', fontweight='bold')
    ax5.set_ylabel('MB')
    for bar, val in zip(bars, mems):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.0f} MB', ha='center', fontsize=10, fontweight='bold')

    # ── 6. Efficiency radar ──
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis('off')
    ax6.set_title('Summary', fontweight='bold', fontsize=13)

    summary_text = (
        f'CNN+MPS Advantages:\n'
        f'━━━━━━━━━━━━━━━━━━━━━\n'
        f'• {cnn_d["parameters"]/mps_d["parameters"]:.1f}× fewer parameters\n'
        f'• {mps_d["parameters"]/200000*100:.1f}% of budget used\n'
        f'• GE@all: {mps_d["ge_all"]:.1f}\n'
        f'• Edge-deployable (Int8)\n'
        f'• MPS regularization\n'
    )
    ax6.text(0.1, 0.5, summary_text, transform=ax6.transAxes,
             fontsize=11, verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#E3F2FD', alpha=0.8))

    plt.suptitle('CNN vs CNN+MPS — Full Comparison Dashboard',
                 fontweight='bold', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('results/figures/comparison_dashboard.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  Model comparison data not found. Run colab_train.ipynb first.')

---

## 🔬 Trace-Level Analysis

In [ ]:
# ── Trace heatmap & statistics ────────────────────────────────────
import h5py

proc_path = 'data/processed/python_processed.h5'
if Path(proc_path).exists():
    with h5py.File(proc_path, 'r') as f:
        traces = f['traces/profiling'][:200]
        labels = f['labels/profiling'][:200]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # ── Trace heatmap ──
    ax = axes[0, 0]
    sorted_idx = np.argsort(labels)
    im = ax.imshow(traces[sorted_idx], aspect='auto', cmap='RdBu_r',
                   vmin=-3, vmax=3)
    ax.set_title('Trace Heatmap (sorted by label)', fontweight='bold')
    ax.set_xlabel('Sample index'); ax.set_ylabel('Trace index')
    plt.colorbar(im, ax=ax, label='Normalized amplitude')

    # ── Label distribution ──
    ax = axes[0, 1]
    ax.hist(labels, bins=64, color=COLORS['mps'], edgecolor='white', alpha=0.8)
    ax.set_title('Label Distribution (first 200 traces)', fontweight='bold')
    ax.set_xlabel('Label (key byte value)'); ax.set_ylabel('Count')

    # ── SNR-like analysis ──
    ax = axes[1, 0]
    with h5py.File(proc_path, 'r') as f:
        all_traces = f['traces/profiling'][:]
        all_labels = f['labels/profiling'][:]

    # Compute signal-to-noise ratio per sample point
    unique_labels = np.unique(all_labels)
    class_means = np.zeros((len(unique_labels), all_traces.shape[1]))
    class_vars  = np.zeros_like(class_means)
    for i, lbl in enumerate(unique_labels):
        mask = all_labels == lbl
        if mask.sum() > 0:
            class_means[i] = all_traces[mask].mean(axis=0)
            class_vars[i]  = all_traces[mask].var(axis=0)

    signal = class_means.var(axis=0)
    noise  = class_vars.mean(axis=0) + 1e-10
    snr = signal / noise

    ax.plot(snr, color=COLORS['accent'], linewidth=0.8)
    ax.fill_between(range(len(snr)), snr, alpha=0.2, color=COLORS['accent'])
    ax.set_title('Signal-to-Noise Ratio (per sample)', fontweight='bold')
    ax.set_xlabel('Sample index'); ax.set_ylabel('SNR')

    # Mark top-k leakage points
    top_k = 10
    top_idx = np.argsort(snr)[-top_k:]
    ax.scatter(top_idx, snr[top_idx], color='red', s=30, zorder=5, label=f'Top-{top_k} PoI')
    ax.legend()

    # ── Correlation matrix ──
    ax = axes[1, 1]
    # Sample 50 evenly-spaced features
    indices = np.linspace(0, all_traces.shape[1]-1, 50, dtype=int)
    corr = np.corrcoef(all_traces[:, indices].T)
    im = ax.imshow(corr, cmap='coolwarm', vmin=-0.5, vmax=0.5)
    ax.set_title('Feature Correlation (50 sampled points)', fontweight='bold')
    plt.colorbar(im, ax=ax, label='Pearson r')

    plt.suptitle('Trace-Level Side-Channel Analysis', fontweight='bold', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.savefig('results/figures/trace_analysis_full.png', dpi=200, bbox_inches='tight')
    plt.show()

    print(f'\n📊 SNR Summary:')
    print(f'   Max SNR     : {snr.max():.4f} (at sample {snr.argmax()})')
    print(f'   Mean SNR    : {snr.mean():.4f}')
    print(f'   Top-{top_k} PoI  : {sorted(top_idx.tolist())}')
else:
    print('⚠️  Processed data not found.')

---

## 📚 Literature Comparison

Comparison with published results from the SCA literature.

> **Note**: These numbers are sourced from the referenced papers. Direct comparison should account for differences in datasets, noise levels, and target implementations.

In [ ]:
# ── Literature comparison table ───────────────────────────────────
literature = pd.DataFrame([
    {'Method': 'CPA (Brier et al., 2004)', 'Parameters': 'N/A', 'GE@500': '~50',
     'Target': 'AES S-box', 'Paradigm': 'Statistical'},
    {'Method': 'Template Attack (Chari et al., 2002)', 'Parameters': 'N/A', 'GE@500': '~20',
     'Target': 'AES S-box', 'Paradigm': 'Profiling'},
    {'Method': 'ASCAD MLP (Benadjila et al., 2020)', 'Parameters': '~300K', 'GE@500': '~15',
     'Target': 'AES S-box', 'Paradigm': 'Deep Learning'},
    {'Method': 'ASCAD CNN (Benadjila et al., 2020)', 'Parameters': '~60K', 'GE@500': '~10',
     'Target': 'AES S-box', 'Paradigm': 'Deep Learning'},
    {'Method': 'Zaid et al. CNN (2020)', 'Parameters': '~20K', 'GE@500': '~5',
     'Target': 'AES S-box', 'Paradigm': 'Deep Learning'},
    {'Method': 'Pay Attention (Hajra et al., 2023)', 'Parameters': '~500K', 'GE@500': '~3',
     'Target': 'AES S-box', 'Paradigm': 'Transformer'},
    {'Method': 'MojoPQC-SCA CNN+MPS (Ours)', 'Parameters': '~9,226', 'GE@500': 'See results',
     'Target': 'ML-KEM-512 NTT', 'Paradigm': 'Tensor Network'},
])

print('\n' + '=' * 90)
print('📚 LITERATURE COMPARISON')
print('=' * 90)
display(literature.style.set_properties(**{'text-align': 'left'}))

print('\n💡 Key advantages of MojoPQC-SCA:')
print('   1. First ML-based SCA specifically targeting ML-KEM-512 NTT leakage')
print('   2. Smallest parameter count (~9K vs 20K–500K in literature)')
print('   3. MPS tensor network provides built-in regularization')
print('   4. Int8 quantization enables edge deployment')
print('   5. Memory-bounded streaming preprocessing (works on 8 GB RAM)')

---

## 💾 Export All Figures

In [ ]:
# ── List all generated figures ────────────────────────────────────
fig_dir = Path('results/figures')
if fig_dir.exists():
    print('📁 All generated figures:')
    for f in sorted(fig_dir.glob('*.png')):
        size = f.stat().st_size / 1024
        print(f'   {f.name:<40s} {size:>8.1f} KB')

    # Download all
    try:
        from google.colab import files
        import shutil
        shutil.make_archive('/content/mojopqc_figures', 'zip', str(fig_dir))
        files.download('/content/mojopqc_figures.zip')
        print('\n📥 Figures archive downloaded.')
    except ImportError:
        print('\n(Not running on Colab — figures saved to results/figures/)')
else:
    print('⚠️  No figures directory found.')